# Evaluation: SAM 3 + SAM 3D Objects against ground truth

This notebook measures how well the pipeline recovers a **single object's 3D pose and size** in a sequence of frames from a **static, calibrated camera** for which the ground truth is known (for example a simulator recording).

For every evaluated frame it reconstructs the object with **SAM 3D Objects**, from the frame and a mask, and compares the result with the ground truth:

| Metric | Meaning |
|---|---|
| `mask_iou` | SAM 3 mask vs ground-truth mask (only with `MASK_SOURCE = "sam3"`) |
| `trans_err_m` | distance between the predicted and true object centre, in metres, in the camera frame |
| `depth_err_m` | predicted minus true distance along the optical axis (negative = predicted too close) |
| `lateral_err_m` | centre error perpendicular to the viewing ray (what you'd see as a shift in the image) |
| `rot_err_deg` | angle between predicted and true orientation, after one fixed alignment of SAM 3D's object axes to the object's axes (see below) |
| `heading_err_deg` | error of the object's forward direction on the ground plane (needs the camera's world pose) |
| `L/W/H_pred_m` | size of the reconstruction along the object's axes, as returned |
| `scale_corr` | true distance / predicted distance. A single image can't tell a small near object from a large far one, so SAM 3D's metric scale can be off |
| `L/W/H_scaled_m` | size after moving the reconstruction to the true distance (`size * scale_corr`): the shape error left once the scale ambiguity is removed |
| `focal_ratio` | focal length SAM 3D implicitly used / true focal length (it estimates the camera itself) |
| `reproj_iou` | silhouette of the reconstruction, projected with the **true** camera intrinsics, vs the ground-truth mask |
| `reproj_iou_own_focal` | same, projected with SAM 3D's own focal length: how well the reconstruction explains the mask |

**Axis alignment.** SAM 3D returns the object in its own canonical axes, which aren't the ground truth's (x forward, y left, z up). The notebook estimates **one** rotation between the two, shared by all frames (the robust mean over frames), and applies it everywhere. The rotation error is therefore the error *after* that single alignment: it measures how consistent the orientation is across frames, and a 180° error means the front and the back were swapped.

**SAM 3D's camera** (`CAMERA_MODE`).
- `"estimate"` (SAM 3D's default): SAM 3D estimates depth and focal length itself with MoGe v1, whose depth is only relative. The distance and size it returns are then not metric, and errors in depth and size include its focal-length and scale errors.
- `"known"`: SAM 3D gets a point map built from the true intrinsics and MoGe's depth, made metric by fitting it to the ground plane (known from the static camera's pose) around the object.

The input format is described in [docs/EVALUATION.md](docs/EVALUATION.md).

**Kernel:** `Python (.venv-sam3d)`. For `MASK_SOURCE = "sam3"`, first run `sam3.ipynb` (kernel `.venv-sam3`) with `VIDEO_PATH` set to the sequence's `rgb/` folder.

# Cell 1: Settings (the only cell you normally need to edit)

In [ ]:
# ---------------- Inputs ----------------
SEQUENCE_DIR = "./sequences/lap_03_turntable"   # rgb/, camera.json, gt.csv, masks/ (see docs/EVALUATION.md)
MASK_SOURCE = "sam3"             # "sam3": masks from sam3.ipynb (SAM3_OUTPUT_DIR) | "gt": ground-truth masks (tests SAM 3D alone)
SAM3_OUTPUT_DIR = "./sam3_output"
SAM3D_REPO = "./sam-3d-objects"
CHECKPOINT_TAG = "hf"

# ---------------- Which frames ----------------
TARGET_FPS = 3              # frames per second to evaluate. With MASK_SOURCE="sam3" the frames SAM 3 processed are used instead
FRAME_RANGE = None          # (first, last) frame numbers, inclusive. None -> the whole sequence
MAX_FRAMES = None           # evaluate at most this many frames (None -> all)
MIN_MASK_FRAC = 0.005       # skip frames where the object covers less than this fraction of the image (ground-truth mask)
SKIP_TRUNCATED = True       # skip frames where the object is cut by the image border

OBJECT_SIZE_LWH = None      # true box size (length, width, height) in m. None -> read from the sequence if available

# ---------------- Camera given to SAM 3D ----------------
CAMERA_MODE = "estimate"    # "estimate": SAM 3D estimates depth and focal length itself (default SAM 3D behaviour)
                            # "known": true intrinsics + metric depth, scaled with the ground plane (needs the world pose)
GROUND_Z = None             # ground height in the world frame, m. None -> camera.json / traj.json "ground_z"

# ---------------- Reconstruction ----------------
SEED = 42
CLEAN_MASKS = True          # SAM 3 masks only: keep the largest blob and fill small holes
RESUME = True               # reuse reconstructions already saved in the output folder (safe to re-run after a crash)
SAVE_PLY = False            # also save each frame's Gaussian splat (~5-10 MB each, in SAM 3D's object coordinates)
MAX_POINTS = 20000          # points kept per frame for the metrics and the overlay
OOM_RETRIES = 3             # on a shared GPU: wait and retry a frame that ran out of memory
OUTPUT_DIR = None           # None -> ./eval_output/<sequence name>_<mask source>_<camera mode>
# ------------------------------------------------

# Cell 2: Environment check and imports

In [ ]:
import os
import sys
import csv
import json
import gc
import time
import platform
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch

print(f"OS: {platform.system()} ({platform.machine()}) | Python {platform.python_version()} | PyTorch {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. SAM 3D Objects needs an NVIDIA GPU (Linux).")
props = torch.cuda.get_device_properties(0)
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB, {free_gb:.1f} GB free")
if free_gb < 20:
    print("WARNING: SAM 3D needs about 20 GB of free VRAM. Loading the model may fail with out-of-memory.")

# Absolute paths (Cell 4 changes the working directory to the SAM 3D repository)
NOTEBOOK_DIR = globals().get("NOTEBOOK_DIR", Path.cwd())
os.chdir(NOTEBOOK_DIR)
assert MASK_SOURCE in ("sam3", "gt"), 'MASK_SOURCE must be "sam3" or "gt"'
assert CAMERA_MODE in ("estimate", "known"), 'CAMERA_MODE must be "estimate" or "known"'
SEQ = Path(SEQUENCE_DIR).resolve()
REPO = Path(SAM3D_REPO).resolve()
SAM3_DIR = Path(SAM3_OUTPUT_DIR).resolve()
OUT = Path(OUTPUT_DIR).resolve() if OUTPUT_DIR else NOTEBOOK_DIR / "eval_output" / f"{SEQ.name}_{MASK_SOURCE}_{CAMERA_MODE}"
REC = OUT / "reconstructions"
REC.mkdir(parents=True, exist_ok=True)

for p in (SEQ / "rgb", SEQ / "camera.json", SEQ / "gt.csv"):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found. See docs/EVALUATION.md for the sequence format.")
if MASK_SOURCE == "sam3" and not (SAM3_DIR / "tracks.csv").exists():
    raise FileNotFoundError(f"{SAM3_DIR / 'tracks.csv'} not found. Run sam3.ipynb on {SEQ / 'rgb'} first, or use MASK_SOURCE = 'gt'.")
config_path = REPO / "checkpoints" / CHECKPOINT_TAG / "pipeline.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"{config_path} not found (see docs/SETUP.md).")

sys.path.insert(0, str(REPO / "notebook"))
sys.path.insert(0, str(REPO))
print(f"Sequence: {SEQ}\nOutput:   {OUT}")

# Cell 3: Load the sequence and the ground truth, choose the frames and the masks

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg"}


def index_images(folder):
    """{frame number: path} for files named by their frame number (e.g. 000123.png)."""
    folder = Path(folder)
    if not folder.is_dir():
        return {}
    return {int(p.stem): p for p in sorted(folder.iterdir()) if p.suffix.lower() in IMG_EXTS and p.stem.isdigit()}


def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


# ---- Camera ----
cam = json.load(open(SEQ / "camera.json"))
K = np.array(cam["K"], dtype=float)
IMG_W, IMG_H = int(cam["width"]), int(cam["height"])
world = cam.get("world") or cam.get("rh_world")          # optional: camera pose in a z-up world frame
R_WC = np.array(world["R_world_from_cam_opencv"], float) if world else None
T_WC = np.array(world["position_m"], float) if world else None
print(f"Camera: {IMG_W}x{IMG_H}, fx={K[0, 0]:.1f}, fy={K[1, 1]:.1f}, cx={K[0, 2]:.1f}, cy={K[1, 2]:.1f}"
      f" | world pose: {'yes' if world else 'no (no heading error / world trajectory)'}")

# ---- Ground truth ----
with open(SEQ / "gt.csv", newline="") as f:
    GT = {int(r["frame"]): r for r in csv.DictReader(f)}
RGB = index_images(SEQ / "rgb")
GT_MASKS = index_images(SEQ / "masks")


def gt_pose(f):
    """True object pose in the OpenCV camera frame: rotation (object -> camera) and centre."""
    r = GT[f]
    R = quat_to_R(num(r["cam_qw"]), num(r["cam_qx"]), num(r["cam_qy"]), num(r["cam_qz"]))
    return R, np.array([num(r["cam_x"]), num(r["cam_y"]), num(r["cam_z"])])


def quat_to_R(w, x, y, z):
    n = np.sqrt(w * w + x * x + y * y + z * z)
    w, x, y, z = w / n, x / n, y / n, z / n
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def load_gt_mask(f):
    p = GT_MASKS.get(f)
    return np.array(Image.open(p).convert("L")) > 127 if p else np.zeros((IMG_H, IMG_W), bool)


# Frame rate, from the time column
times = sorted((f, num(r.get("sim_time_s", r.get("time_s")))) for f, r in GT.items())
dts = np.diff([t for _, t in times if t is not None])
FPS = 1 / np.median(dts) if len(dts) else None
print(f"Ground truth: {len(GT)} frames, {len(RGB)} images, {len(GT_MASKS)} masks" + (f", {FPS:.2f} fps" if FPS else ""))

# True object size
SIZE_GT = np.array(OBJECT_SIZE_LWH, float) if OBJECT_SIZE_LWH else None
if SIZE_GT is None and (SEQ / "traj.json").exists():
    ext = json.load(open(SEQ / "traj.json")).get("bbox_extent")       # CARLA half-sizes
    SIZE_GT = 2 * np.array(ext, float) if ext else None
# Ground plane, for CAMERA_MODE = "known"
if GROUND_Z is None:
    GROUND_Z = (world or {}).get("ground_z")
if GROUND_Z is None and (SEQ / "traj.json").exists():
    GROUND_Z = json.load(open(SEQ / "traj.json")).get("ground_z")
if GROUND_Z is None and SIZE_GT is not None and "world_z" in next(iter(GT.values())):
    # Fallback: the object rests on the ground, so the ground is at (box centre z) - H/2
    zs = [num(r["world_z"]) for r in GT.values() if num(r.get("world_z")) is not None]
    GROUND_Z = float(np.median(zs)) - SIZE_GT[2] / 2 if zs else None
if CAMERA_MODE == "known":
    print(f"Ground height: {GROUND_Z}")
if CAMERA_MODE == "known" and (R_WC is None or GROUND_Z is None):
    raise ValueError('CAMERA_MODE = "known" needs the camera world pose and the ground height (GROUND_Z).')

print("True size (L, W, H):", np.round(SIZE_GT, 3).tolist() if SIZE_GT is not None else "unknown (size metrics skipped)")


def gt_usable(f):
    r = GT[f]
    if r.get("visibility", "visible") != "visible":
        return False
    if SKIP_TRUNCATED and int(num(r.get("truncated")) or 0):
        return False
    frac = num(r.get("mask_frac"))
    if frac is None:
        frac = load_gt_mask(f).mean()
    return frac >= MIN_MASK_FRAC


def in_range(f):
    return FRAME_RANGE is None or FRAME_RANGE[0] <= f <= FRAME_RANGE[1]


def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def clean_mask(m):
    """Keep the largest connected blob and fill small holes."""
    m8 = m.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m8, connectivity=8)
    if n > 2:
        m8 = (labels == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    k = max(3, int(round(min(m.shape) * 0.01)) | 1)
    return cv2.morphologyEx(m8, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8)).astype(bool)


# ---- Frames to evaluate ----
FRAME_INFO = {}       # frame -> dict of per-frame info (mask source, SAM 3 IoU, ...)
MASKS = {}            # frame -> mask given to SAM 3D

if MASK_SOURCE == "gt":
    step = max(1, int(round(FPS / TARGET_FPS))) if FPS else 1
    sampled = [f for f in sorted(GT) if f in RGB and in_range(f)][::step]
    print(f"Sampling 1 frame every {step} (~{FPS / step:.2f} fps)" if FPS else "Using every frame")
else:
    run_info = json.load(open(SAM3_DIR / "run_info.json"))
    with open(SAM3_DIR / "tracks.csv", newline="") as f:
        tracks = [r for r in csv.DictReader(f) if r["mask_file"]]
    processed = sorted(int(p.stem.split("_")[1]) for p in (SAM3_DIR / "frames").glob("frame_*.png"))
    # SAM 3 frame index -> frame number of the sequence
    if run_info.get("source_frames"):
        to_seq = dict(enumerate(run_info["source_frames"]))
    else:
        to_seq = {i: run_info["start_frame"] + i * run_info["interval"] for i in processed}
    by_seq = {}
    for r in tracks:
        by_seq.setdefault(to_seq[int(r["frame"])], []).append(r)
    sampled = [to_seq[i] for i in processed if to_seq[i] in GT and in_range(to_seq[i])]
    print(f"SAM 3 output: {len(processed)} frames processed (~{run_info['output_fps']:.2f} fps), prompts {run_info['prompts']}")

EVAL = [f for f in sampled if gt_usable(f)]
if MAX_FRAMES:
    EVAL = EVAL[:MAX_FRAMES]
print(f"{len(sampled)} sampled frames, {len(EVAL)} usable (object visible, mask >= {MIN_MASK_FRAC:.1%}"
      f"{', not truncated' if SKIP_TRUNCATED else ''})")
if not EVAL:
    raise RuntimeError("No frame left. Relax MIN_MASK_FRAC / SKIP_TRUNCATED / FRAME_RANGE.")

for f in EVAL:
    gt_mask = load_gt_mask(f)
    info = {"frame": f, "gt_mask_frac": float(gt_mask.mean())}
    if MASK_SOURCE == "gt":
        MASKS[f] = gt_mask
    else:
        # Pick the SAM 3 detection that best overlaps the true object (the others are counted as extra detections)
        best, best_iou, best_id = None, 0.0, None
        dets = by_seq.get(f, [])
        for r in dets:
            m = np.array(Image.open(SAM3_DIR / r["mask_file"]).convert("L")) > 127
            if m.shape != (IMG_H, IMG_W):
                m = cv2.resize(m.astype(np.float32), (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR) > 0.5
            iou = mask_iou(m, gt_mask)
            if best is None or iou > best_iou:
                best, best_iou, best_id = m, iou, int(r["object_id"])
        info.update(sam3_detections=len(dets), sam3_object_id=best_id, mask_iou=best_iou)
        if best is not None and best.any():
            MASKS[f] = clean_mask(best) if CLEAN_MASKS else best
    FRAME_INFO[f] = info

if MASK_SOURCE == "sam3":
    ious = np.array([FRAME_INFO[f]["mask_iou"] for f in EVAL])
    missed = [f for f in EVAL if f not in MASKS]
    extra = sum(max(0, FRAME_INFO[f]["sam3_detections"] - 1) for f in EVAL)
    print(f"SAM 3 masks vs ground truth: mean IoU {ious.mean():.3f}, median {np.median(ious):.3f}, "
          f"min {ious.min():.3f} | missed in {len(missed)} frames | {extra} extra detections")

todo = [f for f in EVAL if f in MASKS and not (RESUME and (REC / f"frame_{f:06d}.npz").exists())]
print(f"\nTo reconstruct: {len(todo)} frames ({len(EVAL) - len(todo)} already done or without a mask). "
      f"Estimated time: ~{(60 + 7.5 * len(todo)) / 60:.0f} min on an RTX 4090 (about 1 min to load, ~7 s per frame).")

# ---- Show a few frames: green = true mask, magenta = mask given to SAM 3D ----
show = EVAL[:: max(1, len(EVAL) // 4)][:4]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 3.5), squeeze=False)
for ax, f in zip(axes[0], show):
    img = np.array(Image.open(RGB[f]).convert("RGB"))
    for m, color in ((load_gt_mask(f), (0, 255, 0)), (MASKS.get(f, np.zeros((IMG_H, IMG_W), bool)), (255, 0, 255))):
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, color, 3)
    ys, xs = np.where(load_gt_mask(f))
    pad = 80
    ax.imshow(img[max(0, ys.min() - pad):ys.max() + pad, max(0, xs.min() - pad):xs.max() + pad])
    title = f"frame {f}" + (f" | IoU {FRAME_INFO[f]['mask_iou']:.2f}" if MASK_SOURCE == "sam3" else "")
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

# Cell 4: Load SAM 3D Objects (skipped if every frame is already reconstructed)

In [ ]:
if "inference" in globals():
    del inference
gc.collect()
torch.cuda.empty_cache()

if todo:
    os.chdir(REPO)                       # the repository code expects to run from its root
    from inference import Inference
    print("Loading SAM 3D Objects...")
    t0 = time.time()
    for attempt in range(OOM_RETRIES + 1):
        try:
            inference = Inference(str(config_path), compile=False)
            break
        except Exception as e:                       # on a shared GPU another job may hold the memory for a while
            gc.collect(); torch.cuda.empty_cache()
            if "out of memory" not in str(e).lower() or attempt == OOM_RETRIES:
                raise
            print(f"Out of GPU memory while loading, retrying in 30 s ({attempt + 1}/{OOM_RETRIES})")
            time.sleep(30)
    print(f"Model ready in {time.time() - t0:.0f} s | VRAM in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("Nothing to reconstruct: all frames are already in", REC)

# Cell 5: Reconstruct every frame

Each frame is saved as `reconstructions/frame_NNNNNN.npz` as soon as it's done (pose, a subsample of the points with their colours, the mask used), so an interrupted run can be resumed by running Cells 1-5 again.

In [ ]:
SH_C0 = 0.28209479177387814           # spherical-harmonics constant: colour = 0.5 + SH_C0 * f_dc
rng = np.random.default_rng(0)


def to_np(x):
    return x.detach().float().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)


# Ground-plane depth of every pixel: where its viewing ray hits z = GROUND_Z (inf if it doesn't)
if CAMERA_MODE == "known":
    _u, _v = np.meshgrid(np.arange(IMG_W) + 0.5, np.arange(IMG_H) + 0.5)
    _rays = np.stack([(_u - K[0, 2]) / K[0, 0], (_v - K[1, 2]) / K[1, 1], np.ones_like(_u)], -1)   # z = 1 (OpenCV)
    _dz = _rays @ R_WC[2]                                       # world z component of each ray
    with np.errstate(divide="ignore", invalid="ignore"):
        GROUND_DEPTH = np.where(_dz < -1e-6, (GROUND_Z - T_WC[2]) / _dz, np.inf)
    HFOV_DEG = float(np.degrees(2 * np.arctan(IMG_W / (2 * K[0, 0]))))


def known_camera_pointmap(img, mask):
    """Metric point map for SAM 3D: MoGe depth with the true field of view, scaled so the ground
    around the object lies on the true ground plane, back-projected with the true intrinsics.
    Returned in PyTorch3D camera coordinates (x left, y up, z forward), H x W x 3."""
    depth_model = inference._pipeline.depth_model
    x = torch.from_numpy(img).float().permute(2, 0, 1).div(255).to(depth_model.device)
    with torch.no_grad():
        depth = depth_model.model.infer(x, fov_x=HFOV_DEG, force_projection=True)["depth"].float().cpu().numpy()
    # Ground pixels near the object (box around it, 1.5x its size each side), not on the object itself
    ys, xs = np.where(mask)
    h, w = ys.ptp() + 1, xs.ptp() + 1
    y0, y1 = max(0, ys.min() - int(1.5 * h)), min(IMG_H, ys.max() + int(1.5 * h))
    x0, x1 = max(0, xs.min() - int(1.5 * w)), min(IMG_W, xs.max() + int(1.5 * w))
    near = np.zeros_like(mask); near[y0:y1, x0:x1] = True
    obj = cv2.dilate(mask.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
    sel = near & ~obj & np.isfinite(depth) & (depth > 0) & np.isfinite(GROUND_DEPTH)
    scale = float(np.median(GROUND_DEPTH[sel] / depth[sel]))
    z = scale * depth
    pts = np.stack([-_rays[..., 0] * z, -_rays[..., 1] * z, z], -1).astype(np.float32)   # OpenCV -> PyTorch3D
    return torch.from_numpy(pts), scale, int(sel.sum())


if todo and "inference" not in globals():
    raise RuntimeError("The model is not loaded: run Cell 4 first.")

t_start = time.time()
for i, f in enumerate(todo):
    img = np.array(Image.open(RGB[f]).convert("RGB"))
    mask = MASKS[f]
    out = None
    for attempt in range(OOM_RETRIES + 1):
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        try:
            if CAMERA_MODE == "known":
                pointmap, depth_scale, n_ground = known_camera_pointmap(img, mask)
                out = inference(img, mask, seed=SEED, pointmap=pointmap)
            else:
                depth_scale, n_ground = np.nan, 0
                out = inference(img, mask, seed=SEED)
            break
        except Exception as e:                       # torch and spconv report OOM differently
            gc.collect(); torch.cuda.empty_cache()
            if "out of memory" not in str(e).lower():
                print(f"frame {f}: failed ({type(e).__name__}: {e})")
                break
            if attempt < OOM_RETRIES:
                print(f"frame {f}: out of GPU memory, retrying in 20 s ({attempt + 1}/{OOM_RETRIES})")
                time.sleep(20)
            else:
                print(f"frame {f}: out of GPU memory, skipped (run Cell 5 again later to retry)")
    if out is None:
        continue
    dt = time.time() - t0

    g = out["gaussian"][0]
    xyz = to_np(g.get_xyz)
    opacity = to_np(g.get_opacity).reshape(-1)
    rgb = np.clip(0.5 + SH_C0 * to_np(g._features_dc).reshape(len(xyz), -1)[:, :3], 0, 1)
    keep = np.flatnonzero(opacity > 0.5)
    if len(keep) < 100:
        keep = np.flatnonzero(opacity > 0.1)
    if len(keep) > MAX_POINTS:
        keep = rng.choice(keep, MAX_POINTS, replace=False)

    np.savez_compressed(
        REC / f"frame_{f:06d}.npz", frame=f, seconds=dt,
        quat_wxyz=to_np(out["rotation"]).reshape(4), translation=to_np(out["translation"]).reshape(3),
        scale=to_np(out["scale"]).reshape(3), xyz_local=xyz[keep].astype(np.float32), rgb=rgb[keep].astype(np.float16),
        mask=np.packbits(mask), mask_shape=np.array(mask.shape), depth_scale=depth_scale, ground_pixels=n_ground)
    if SAVE_PLY:
        g.save_ply(str(REC / f"frame_{f:06d}.ply"))

    done = i + 1
    eta = (time.time() - t_start) / done * (len(todo) - done)
    print(f"[{done}/{len(todo)}] frame {f}: {dt:.1f} s" + (f", depth scale {depth_scale:.2f}" if CAMERA_MODE == "known" else "") + f", peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
          f" | ETA {eta / 60:.1f} min")

if todo:
    print(f"\nReconstructed {len(todo)} frames in {(time.time() - t_start) / 60:.1f} min")
    del inference
    gc.collect(); torch.cuda.empty_cache()
    print(f"Model freed | VRAM in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
os.chdir(NOTEBOOK_DIR)

# Cell 6: Compare with the ground truth

SAM 3D returns, for each frame, a rotation `q`, a translation `t` and a scale `s` that place its object points `p` in a PyTorch3D camera frame (x left, y up, z forward): `p_cam = (s * p) @ R(q) + t` (row vectors). Converted to the OpenCV camera frame of the ground truth (x right, y down, z forward), with `F = diag(-1, -1, 1)`:

`p_cv = M (s p) + F t`, where `M = F R(q)^T`

The cell checks this formula against PyTorch3D on the first frame before using it.

In [ ]:
import pandas as pd
from pytorch3d.transforms import Transform3d, quaternion_to_matrix

FLIP = np.diag([-1.0, -1.0, 1.0])          # PyTorch3D camera <-> OpenCV camera


def angle_deg(R):
    return float(np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1))))


def nearest_rotation(A):
    U, _, Vt = np.linalg.svd(A)
    return U @ np.diag([1, 1, np.sign(np.linalg.det(U @ Vt))]) @ Vt


# ---- Load the reconstructions ----
recs = {}
for f in EVAL:
    p = REC / f"frame_{f:06d}.npz"
    if not p.exists():
        continue
    d = np.load(p)
    R_l2c = quat_to_R(*d["quat_wxyz"])
    recs[f] = {"M": FLIP @ R_l2c.T, "s": float(d["scale"][0]), "t": FLIP @ d["translation"],
               "xyz": d["xyz_local"].astype(float), "rgb": d["rgb"].astype(float), "seconds": float(d["seconds"]), "raw": d}
print(f"{len(recs)}/{len(EVAL)} frames reconstructed")
if not recs:
    raise RuntimeError("No reconstruction found: run Cells 4-5.")

# ---- Check the conversion against PyTorch3D (first frame) ----
d = next(iter(recs.values()))["raw"]
pts = torch.tensor(d["xyz_local"][:500], dtype=torch.float64)[None]
tf = (Transform3d(dtype=torch.float64).scale(torch.tensor(d["scale"], dtype=torch.float64)[None])
      .rotate(quaternion_to_matrix(torch.tensor(d["quat_wxyz"], dtype=torch.float64)[None]))
      .translate(torch.tensor(d["translation"], dtype=torch.float64)[None]))
ref_cv = tf.transform_points(pts)[0].numpy() @ FLIP
r0 = next(iter(recs.values()))
mine = (r0["M"] @ (r0["s"] * d["xyz_local"][:500].astype(float)).T).T + r0["t"]
assert np.allclose(mine, ref_cv, atol=1e-4 * (1 + np.abs(ref_cv).max())), "pose conversion does not match PyTorch3D"
print("Pose conversion checked against PyTorch3D.")

# ---- One fixed rotation from SAM 3D's object axes to the ground truth's object axes ----
GT_POSE = {f: gt_pose(f) for f in recs}
C_f = {f: GT_POSE[f][0].T @ recs[f]["M"] for f in recs}        # SAM 3D axes -> true object axes, per frame
C = nearest_rotation(sum(C_f.values()))
for _ in range(5):                                              # robust mean: ignore frames more than 45 deg off
    inliers = [f for f in C_f if angle_deg(C_f[f] @ C.T) < 45]
    if inliers:
        C = nearest_rotation(sum(C_f[f] for f in inliers))
print(f"Axis alignment: {len(inliers)}/{len(C_f)} frames within 45 deg of the common alignment")


def heading_deg(R_obj_to_cam):
    fwd = R_WC @ R_obj_to_cam[:, 0]
    return float(np.degrees(np.arctan2(fwd[1], fwd[0])))


def silhouette(points_cv, s, z_med, KK=None):
    """Approximate silhouette of a point cloud projected with intrinsics KK (default: the true ones)."""
    KK = K if KK is None else KK
    sil = np.zeros((IMG_H, IMG_W), np.uint8)
    front = points_cv[:, 2] > 0.01
    uv = (KK @ (points_cv[front] / points_cv[front, 2:3]).T).T[:, :2]
    r = max(1, int(np.ceil(0.75 * KK[0, 0] * (s / 64) / z_med)))     # about one SAM 3D voxel
    for u, v in np.round(uv).astype(int):
        if -r <= u < IMG_W + r and -r <= v < IMG_H + r:
            cv2.circle(sil, (u, v), r, 1, -1)
    return cv2.morphologyEx(sil, cv2.MORPH_CLOSE, np.ones((2 * r + 1, 2 * r + 1), np.uint8)).astype(bool)


rows = []
for f, r in recs.items():
    R_gt, t_gt = GT_POSE[f]
    pts_obj = (C @ (r["s"] * r["xyz"]).T).T                    # points in the true object axes
    lo, hi = np.percentile(pts_obj, [1, 99], axis=0)
    size, c_obj = hi - lo, (lo + hi) / 2
    R_pred = r["M"] @ C.T                                       # true object axes -> camera
    c_pred = R_pred @ c_obj + r["t"]
    r.update(R_pred=R_pred, center=c_pred, size=size, pts_cv=(R_pred @ (pts_obj - c_obj).T).T + c_pred)

    # Focal length SAM 3D implicitly used: the one that makes its points cover the mask it was given
    m_used = np.unpackbits(r["raw"]["mask"])[: int(np.prod(r["raw"]["mask_shape"]))].reshape(r["raw"]["mask_shape"]).astype(bool)
    ys, xs = np.where(m_used)
    pc = r["pts_cv"][r["pts_cv"][:, 2] > 0.01]
    xn, yn = pc[:, 0] / pc[:, 2], pc[:, 1] / pc[:, 2]
    f_eff = np.mean([(xs.max() - xs.min()) / np.ptp(xn), (ys.max() - ys.min()) / np.ptp(yn)])
    r["K_eff"] = np.array([[f_eff, 0, K[0, 2]], [0, f_eff, K[1, 2]], [0, 0, 1]])
    k = np.linalg.norm(t_gt) / np.linalg.norm(c_pred)          # metric-scale correction along the viewing ray

    err = c_pred - t_gt
    ray = t_gt / np.linalg.norm(t_gt)
    row = {**FRAME_INFO[f],
           "true_dist_m": np.linalg.norm(t_gt), "pred_dist_m": np.linalg.norm(c_pred),
           "trans_err_m": np.linalg.norm(err), "depth_err_m": c_pred[2] - t_gt[2],
           "lateral_err_m": np.linalg.norm(err - err.dot(ray) * ray),
           "rot_err_deg": angle_deg(R_pred.T @ R_gt),
           "L_pred_m": size[0], "W_pred_m": size[1], "H_pred_m": size[2],
           "scale_corr": k, "L_scaled_m": k * size[0], "W_scaled_m": k * size[1], "H_scaled_m": k * size[2],
           "focal_ratio": f_eff / K[0, 0],
           "reproj_iou": mask_iou(silhouette(r["pts_cv"], r["s"], c_pred[2]), load_gt_mask(f)),
           "reproj_iou_own_focal": mask_iou(silhouette(r["pts_cv"], r["s"], c_pred[2], r["K_eff"]), load_gt_mask(f)),
           "seconds": r["seconds"]}
    if R_WC is not None:
        row["heading_err_deg"] = (heading_deg(R_pred) - heading_deg(R_gt) + 180) % 360 - 180
    rows.append(row)

df = pd.DataFrame(rows).set_index("frame")
df.to_csv(OUT / "metrics.csv", float_format="%.4f")

# ---- Summary ----
cols = ["mask_iou", "trans_err_m", "depth_err_m", "lateral_err_m", "rot_err_deg", "heading_err_deg",
        "reproj_iou", "reproj_iou_own_focal", "focal_ratio", "scale_corr",
        "L_pred_m", "W_pred_m", "H_pred_m", "L_scaled_m", "W_scaled_m", "H_scaled_m", "seconds"]
cols = [c for c in cols if c in df]
summary = df[cols].agg(["mean", "median", "min", "max"]).T
summary["true"] = np.nan
if SIZE_GT is not None:
    summary.loc[["L_pred_m", "W_pred_m", "H_pred_m"], "true"] = SIZE_GT
    summary.loc[["L_scaled_m", "W_scaled_m", "H_scaled_m"], "true"] = SIZE_GT
if "heading_err_deg" in df:
    summary.loc["|heading_err_deg|"] = df["heading_err_deg"].abs().agg(["mean", "median", "min", "max"]).tolist() + [np.nan]
print(f"\n{len(df)} frames, true distance {df.true_dist_m.min():.1f}-{df.true_dist_m.max():.1f} m\n")
print(summary.round(3).to_string())
print(f"\nFrames with rotation error > 90 deg (front/back swapped or similar): {int((df.rot_err_deg > 90).sum())}")

json.dump({"sequence": str(SEQ), "mask_source": MASK_SOURCE, "seed": SEED, "frames": len(df),
           "settings": {"camera_mode": CAMERA_MODE, "target_fps": TARGET_FPS, "frame_range": FRAME_RANGE, "min_mask_frac": MIN_MASK_FRAC,
                        "skip_truncated": SKIP_TRUNCATED, "clean_masks": CLEAN_MASKS},
           "true_size_lwh": SIZE_GT.tolist() if SIZE_GT is not None else None,
           "axis_alignment_sam3d_to_object": C.tolist(),
           "summary": json.loads(summary.to_json(orient="index"))},
          open(OUT / "summary.json", "w"), indent=2)
print(f"\nSaved: {OUT / 'metrics.csv'}, {OUT / 'summary.json'}")

# Cell 7: Plots

In [ ]:
fr = df.index.values
n_plots = 4 if "mask_iou" in df else 3
fig, axes = plt.subplots(n_plots, 1, figsize=(11, 2.6 * n_plots), sharex=True)
ax = axes[0]
ax.plot(fr, df.trans_err_m, "o-", label="centre error")
ax.plot(fr, df.depth_err_m, "o-", label="depth error (signed)")
ax.plot(fr, df.lateral_err_m, "o-", label="lateral error")
ax.axhline(0, color="gray", lw=0.5); ax.set_ylabel("m"); ax.legend(); ax.set_title("Position")
ax = axes[1]
ax.plot(fr, df.rot_err_deg, "o-", label="rotation error")
if "heading_err_deg" in df:
    ax.plot(fr, df.heading_err_deg.abs(), "o-", label="|heading error|")
ax.set_ylabel("deg"); ax.legend(); ax.set_title("Orientation")
ax = axes[2]
for c, k in zip(("L_scaled_m", "W_scaled_m", "H_scaled_m"), range(3)):
    line, = ax.plot(fr, df[c], "o-", label=c[0])
    if SIZE_GT is not None:
        ax.axhline(SIZE_GT[k], color=line.get_color(), ls="--", lw=1)
ax.set_ylabel("m"); ax.legend(); ax.set_title("Size, rescaled to the true distance (dashed: true)")
if "mask_iou" in df:
    ax = axes[3]
    ax.plot(fr, df.mask_iou, "o-", label="SAM 3 mask vs true")
    ax.plot(fr, df.reproj_iou, "o-", label="reprojected reconstruction vs true")
    ax.set_ylim(0, 1); ax.legend(); ax.set_title("IoU")
else:
    axes[2].set_xlabel("frame")
axes[-1].set_xlabel("frame")
plt.tight_layout(); plt.savefig(OUT / "errors.png", dpi=120); plt.show()

# ---- Trajectory seen from above ----
if R_WC is not None:
    to_world = lambda p: R_WC @ p + T_WC
    gt_w = np.array([to_world(GT_POSE[f][1]) for f in recs])
    pr_w = np.array([to_world(recs[f]["center"]) for f in recs])
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(gt_w[:, 0], gt_w[:, 1], "g.-", label="true centre")
    ax.plot(pr_w[:, 0], pr_w[:, 1], "m.", label="predicted centre")
    for a, b in zip(gt_w, pr_w):
        ax.plot([a[0], b[0]], [a[1], b[1]], color="gray", lw=0.5)
    for f, a, b in zip(recs, gt_w, pr_w):          # forward direction
        for R, p, c in ((GT_POSE[f][0], a, "g"), (recs[f]["R_pred"], b, "m")):
            d = (R_WC @ R[:, 0])[:2]
            ax.arrow(p[0], p[1], 0.5 * d[0], 0.5 * d[1], color=c, width=0.02, alpha=0.6)
    ax.plot(*T_WC[:2], "k^", ms=10, label="camera")
    view = R_WC[:2, 2] / (np.linalg.norm(R_WC[:2, 2]) + 1e-9)
    ax.arrow(T_WC[0], T_WC[1], 2 * view[0], 2 * view[1], color="k", width=0.05)
    ax.set_aspect("equal"); ax.legend(); ax.set_xlabel("world x (m)"); ax.set_ylabel("world y (m)")
    ax.set_title("Seen from above (arrows: forward direction)")
    plt.tight_layout(); plt.savefig(OUT / "trajectory_top.png", dpi=120); plt.show()

# Cell 8: Digital twin overlay

Each reconstruction is drawn over the real image: coloured points = SAM 3D's object, magenta box = its predicted pose and size, green box = ground truth (true camera). SAM 3D's object is projected with the focal length SAM 3D itself assumed (`focal_ratio` in the metrics), so it lines up with the image even when its metric scale is off. All frames are written to `overlay.mp4`.

In [ ]:
EDGES = [(0, 1), (1, 3), (3, 2), (2, 0), (4, 5), (5, 7), (7, 6), (6, 4), (0, 4), (1, 5), (2, 6), (3, 7)]


def box_corners(R, t, size):
    signs = np.array([[sx, sy, sz] for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)], float)
    return (R @ (signs * size / 2).T).T + t


def project(p, KK=None):
    KK = K if KK is None else KK
    p = p[p[:, 2] > 0.01]
    return (KK @ (p / p[:, 2:3]).T).T[:, :2]


def draw_box(img, corners, color, w, KK=None):
    if (corners[:, 2] <= 0.01).any():
        return
    uv = np.round(project(corners, KK)).astype(int)
    for a, b in EDGES:
        cv2.line(img, tuple(uv[a]), tuple(uv[b]), color, w, cv2.LINE_AA)
    front = np.round(project(np.array([corners[[4, 5, 6, 7]].mean(0)]), KK)).astype(int)[0]   # +x face centre
    cv2.circle(img, tuple(front), 2 * w, color, -1)


def overlay(f):
    img = np.array(Image.open(RGB[f]).convert("RGB"))
    r = recs[f]
    order = np.argsort(-r["pts_cv"][:, 2])                    # far points first
    uv = np.round(project(r["pts_cv"][order], r["K_eff"])).astype(int)
    col = (r["rgb"][order] * 255).astype(np.uint8)
    ok = (uv[:, 0] >= 0) & (uv[:, 0] < IMG_W) & (uv[:, 1] >= 0) & (uv[:, 1] < IMG_H)
    twin = img.copy()
    twin[uv[ok, 1], uv[ok, 0]] = col[ok]
    img = cv2.addWeighted(twin, 0.85, img, 0.15, 0)
    lw = max(2, IMG_W // 640)
    if SIZE_GT is not None:
        R_gt, t_gt = GT_POSE[f]
        draw_box(img, box_corners(R_gt, t_gt, SIZE_GT), (0, 255, 0), lw)
    draw_box(img, box_corners(r["R_pred"], r["center"], r["size"]), (255, 0, 255), lw, r["K_eff"])
    row = df.loc[f]
    txt = f"frame {f}  centre err {row.trans_err_m:.2f} m  rot err {row.rot_err_deg:.0f} deg"
    cv2.putText(img, txt, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, IMG_W / 1500, (255, 255, 255), 3, cv2.LINE_AA)
    return img


video_fps = max(1.0, (FPS or 30) / max(1, int(round((FPS or 30) / TARGET_FPS))))
writer = cv2.VideoWriter(str(OUT / "overlay.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), video_fps, (IMG_W, IMG_H))
for f in recs:
    writer.write(cv2.cvtColor(overlay(f), cv2.COLOR_RGB2BGR))
writer.release()
print(f"Saved: {OUT / 'overlay.mp4'}")

show = list(recs)[:: max(1, len(recs) // 6)][:6]
fig, axes = plt.subplots((len(show) + 1) // 2, 2, figsize=(16, 4.6 * ((len(show) + 1) // 2)), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, f in zip(axes.flat, show):
    img = overlay(f)
    ys, xs = np.where(load_gt_mask(f))
    pad = int(0.6 * max(ys.ptp(), xs.ptp()))
    ax.imshow(img[max(0, ys.min() - pad):ys.max() + pad, max(0, xs.min() - pad):xs.max() + pad])
plt.tight_layout(); plt.show()